In [4]:
import ray
import os
from minio import Minio

client = Minio(endpoint='storage:9000',
               access_key=os.getenv('MINIO_ROOT_USER'),
               secret_key=os.getenv('MINIO_ROOT_PASSWORD'),
               secure=False)
ray.init(address=f"ray://ray-head:10001")

Python version:,3.10.14
Ray version:,2.31.0
Dashboard:,http://172.30.0.3:8265


In [5]:
client.list_buckets()

[Bucket('default')]

In [6]:
# check nodes
ray.nodes()

[{'NodeID': '10a34b01706f756dc86bddfb7f376b24c0952d5eed71b091d1d7f972',
  'Alive': True,
  'NodeManagerAddress': '172.30.0.3',
  'NodeManagerHostname': '67d945ebf0cb',
  'NodeManagerPort': 35225,
  'ObjectManagerPort': 38637,
  'ObjectStoreSocketName': '/tmp/ray/session_2024-07-21_04-21-26_954566_1/sockets/plasma_store',
  'RayletSocketName': '/tmp/ray/session_2024-07-21_04-21-26_954566_1/sockets/raylet',
  'MetricsExportPort': 41001,
  'NodeName': '172.30.0.3',
  'RuntimeEnvAgentPort': 52941,
  'DeathReason': 0,
  'DeathReasonMessage': '',
  'alive': True,
  'Resources': {'CPU': 1.0,
   'memory': 685053543.0,
   'object_store_memory': 342526771.0,
   'node:172.30.0.3': 1.0,
   'node:__internal_head__': 1.0},
  'Labels': {'ray.io/node_id': '10a34b01706f756dc86bddfb7f376b24c0952d5eed71b091d1d7f972'}},
 {'NodeID': '5c6f079130e325cde995568d8743eb58ccd881b1b0448952c5fa668d',
  'Alive': True,
  'NodeManagerAddress': '172.30.0.5',
  'NodeManagerHostname': '59b7e64cf439',
  'NodeManagerPort':

In [45]:
# File details
bucket_name = "default"
file_path = "data/European Restaurant Reviews.csv"  # Path to your CSV file
object_name = "European Restaurant Reviews.csv"  # Name of the object to be saved in the bucket

client.fput_object(
        bucket_name, object_name, file_path
    )

In [24]:
res = client.get_object(bucket_name, 'dsad')

S3Error: S3 operation failed; code: NoSuchKey, message: The specified key does not exist., resource: /default/dsad, request_id: 17E43802E6CC707F, host_id: dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8, bucket_name: default, object_name: dsad

In [18]:
output = ''
for record in res.stream():
    output += record.decode()

#[json.loads(_) for _ in output.strip().split('\n')]


In [28]:
rows = [row.split(',') for row in output.strip().split('\r\n')]
headers = rows.pop(0)
import pandas as pd
df = pd.DataFrame.from_records(rows, headers)

ValueError: Length of values (1502) does not match length of index (6)

In [34]:
rows[1501]

['Cuba',
 'Old Square (Plaza Vieja)',
 'Negative',
 'Tourist Trap',
 'Oct 2016 •',
 '"Food and service was awful. Very pretty stop. Good for photos',
 ' bad for food."']

In [40]:
!pip install s3fs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 4.9 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 23.0.1 -> 24.1.2
[notice] To update, run: pip install --upgrade pip


In [44]:
df = pd.read_csv(
    f"s3://default/{object_name}.dss",
    storage_options={
        "key": os.getenv('MINIO_ROOT_USER'),
        "secret": os.getenv('MINIO_ROOT_PASSWORD'),
        "client_kwargs": {"endpoint_url": "http://storage:9000"}
    }
)

FileNotFoundError: default/European Restaurant Reviews.csv.dss

In [43]:
df

,Country,Restaurant Name,Sentiment,Review Title,Review Date,Review
0,France,The Frog at Bercy Village,Negative,Rude manager,May 2024 •,The manager became agressive when I said the c...
1,France,The Frog at Bercy Village,Negative,A big disappointment,Feb 2024 •,"I ordered a beef fillet ask to be done medium,..."
2,France,The Frog at Bercy Village,Negative,Pretty Place with Bland Food,Nov 2023 •,"This is an attractive venue with welcoming, al..."
3,France,The Frog at Bercy Village,Negative,Great service and wine but inedible food,Mar 2023 •,Sadly I used the high TripAdvisor rating too ...
4,France,The Frog at Bercy Village,Negative,Avoid- Worst meal in Rome - possibly ever,Nov 2022 •,From the start this meal was bad- especially g...
...,...,...,...,...,...,...
1497,Cuba,Old Square (Plaza Vieja),Negative,The Tourism Trap,Oct 2016 •,Despite the other reviews saying that this is ...
1498,Cuba,Old Square (Plaza Vieja),Negative,the beer factory,Oct 2016 •,beer is good. food is awfull The only decent...
1499,Cuba,Old Square (Plaza Vieja),Negative,brewery,Oct 2016 •,"for terrible service of a truly comedic level,..."
1500,Cuba,Old Square (Plaza Vieja),Negative,It's nothing exciting over there,Oct 2016 •,We visited the Havana's Club Museum which is l...


In [20]:
object_ids = [f_sleep.remote() for _ in range(10)]
ip_addresses = ray.get(object_ids)

In [23]:
ip_addresses

['172.30.0.5',
 '172.30.0.4',
 '172.30.0.5',
 '172.30.0.4',
 '172.30.0.5',
 '172.30.0.4',
 '172.30.0.5',
 '172.30.0.4',
 '172.30.0.5',
 '172.30.0.4']